## 第17章 PDF文档和Word文档

### 1.使用pypdf库

- 读取PDF文档(PdfReader对象)：`reader = pypdf.PdfReader("filename.pdf")`。
    - 获取页面对象(Page对象)：`pages = reader.pages`，返回一个Page对象列表，Page对象包含页面的属性和内容。
        - 获取页面内容：`text = page.extract_text()`，**注意：获取的文本内容可能包含换行符还有其他符号，需要进一步处理。**
        - 获取页面图片：`images = page.images`，返回一个Image对象列表，`image.name`包含图片名称，`image.data`包含图片数据。

In [ ]:
import pypdf

# 创建PDF文件读取对象
reader = pypdf.PdfReader("res/chapter1.pdf")

# 提取PDF文件中的文本
text = ""
for page in reader.pages:
    text += page.extract_text()
with open("res/chapter1.txt", "w", encoding="utf-8") as f:
    f.write(text)

# 提取PDF文件中的图片
image_num = 0
for i, page in enumerate(reader.pages):
    print(f"Reading page {i+1} - {len(page.images)} images found...")
    try:
        for image in page.images:
            with open(f"res/{image_num}_page{i+1}_{image.name}", "wb") as f:
                f.write(image.data)
            print(f"Wrote {image_num}_page{i+1}_{image.name}...")
            image_num += 1
    except Exception as e:
        print(f"Skipping page {i+1} due to error: {e}")

# 关闭PDF文件读取对象
reader.close()

- 创建PDF文档(PdfWriter对象)：`writer = pypdf.PdfWriter()`，**注意：仅限于复制、合并、裁剪、变换已有PDF页面的内容。**
    - 复制PDF页面：`writer.append("filename.pdf", pages=(0, 5))`，参数pages表示复制的页码范围。**注意：页面添加在文档末尾。**
    - 插入PDF页面：`writer.merge(2, "filename.pdf", pages=(0, 2))`，将filename.pdf的1-2页插入到第2页。
    - 旋转PDF页面：`writer.page[0].rotate(90)`，将第1页旋转90度。
    - 插入空页面：`writer.add_blank_page()`，在文档末尾插入一个空白页。`writer.insert_blank_page(index=x)`，在指定位置插入一个空白页。
    - 添加水印：`writer.merge_page(reader, over=False)`，参数reader为要添加的水印文件，参数over为True时添加覆盖层，为False时添加水印。
    - 加密PDF文档：`writer.encrypt("password", algorithm="AES-256")`，参数password为PDF文档的密码，参数algorithm为加密算法。
    - 解密PDF文档：`reader.is_encrypted`检查是否加密，`writer.decrypt("password")`，参数password为PDF文档的密码。

In [ ]:
import pypdf

# 创建PDF写入器
writer = pypdf.PdfWriter()

# 复制1-5页追加到文档末尾
writer.append("res/chapter1.pdf", (0, 5))

# 在第3页之后追加第6-10页
writer.merge(3, "res/chapter1.pdf", pages=(5, 10))

# 插入空白页
writer.add_blank_page()
writer.insert_blank_page(index=4)

# 从第6页开始，页面旋转90度，并添加水印
reader = pypdf.PdfReader("res/watermark.pdf")
wp = reader.pages[0]
for i in range(5, len(writer.pages)):
    writer.pages[i].rotate(90)
    writer.pages[i].merge_page(wp, over=False)
reader.close()

# 加密文档
writer.encrypt("123456", algorithm="RC4-128")

# 写入文档
with open("res/chapter2.pdf", "wb") as f:
    writer.write(f)

# 关闭
writer.close()

### 2.使用python-docx库

- 读取Word文档：`doc = docx.Document("filename.docx")`。读取Word文档后的对象结构：
    - Document：表示整个Word文档。
    - Paragraph：列表，表示文档中的所有段落。
    - Run：列表，表示段落中格式相同的文本片段。
- 获取文本内容：`text = doc.paragraphs[0].text`，如要获取所有段落的文本，则需要遍历段落列表。
- 设置段落样式：`doc.paragraphs[0].style = "Heading1"`，通过style属性设置段落样式(doc.styles可以获取所有样式)。
- 设置Run样式：`doc.paragraphs[0].runs[0].style = "Heading1 Char"`，设置Run样式时，样式名称必须加上" Char"。
    - 可以通过Run的`bold`、`italic`、`underline`、`strike`等属性来设置样式。
- 添加段落：`doc.add_paragraph("Hello World!")`。
- 段落后添加Run：`doc.paragraphs[0].add_run("Hello World!")`。
- 添加软换行符：`doc.paragraphs[0].add_break()`。
- 添加换页符：`doc.paragraphs[0].add_break(docx.enum.text.WD_BREAK.PAGE)`。
- 添加图片：`doc.add_picture("image.png", width=Inches(1), height=Inches(1))`。
- 保存文档：`doc.save("filename.docx")`。

In [ ]:
import docx
import docx.enum.text
import docx.shared

# 读取Word文档
doc = docx.Document("res/demo.docx")
text = [para.text for para in doc.paragraphs]
print("\n".join(text))

# 设置段落及Run样式
doc.paragraphs[0].style = "Normal"
doc.paragraphs[1].runs[0].style = "Quote Char"
doc.paragraphs[1].runs[1].underline = True
doc.paragraphs[1].runs[3].bold = True

# 保存Word文档
doc.save("res/demo2.docx")

# 创建Word文档
doc2 = docx.Document()

# 添加段落及Run
doc2.add_paragraph("Hello World!", "Title")
p1 = doc2.add_paragraph("This is a second paragraph.")
p2 = doc2.add_paragraph("This is a yet another paragraph.")
p1.add_run(" This text is being added to the second paragraph.")

# 添加分页符
p2.runs[0].add_break(docx.enum.text.WD_BREAK.PAGE)
doc2.add_paragraph("This is a new paragraph after a page break.")

# 添加图片
doc2.add_picture("res/zophie.png", width=docx.shared.Cm(4.76), height=docx.shared.Cm(7.5))

# 保存Word文档
doc2.save("res/demo3.docx")